# Test the time mix code block

Using the reference model, we load the first time mix block, test/validate the forward pass for compilation issues

In [3]:
!ls

00-model-download.ipynb  21-block-test.ipynb  60-timemix-kernel-benchmark.ipynb
12-timemix-test.ipynb	 51-model-test.ipynb  91-hf-builder.test.ipynb


In [1]:
# Configure the parent path to be the proj folder
import sys, os, torch, time
sys.path.append('../../')

# # Cuda debugging
# os.environ["TORCH_USE_CUDA_DSA"] = "1"
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# Import the block classes
from rwkv_block.v7_qwerky.block.qwerky7_time_mix import Qwerky7TimeMix

# File to load
MODEL_FILENAME="rwkv-final.pth"

# Run device, and run dtype to use
RUN_DEVICE="cpu"
RUN_DTYPE=torch.bfloat16
RUN_TMIX_BACKEND="naive"

# Check for cuda device
# if torch.cuda.is_available():
#     RUN_DEVICE="cuda:0"

# Check if the reference weights exists
assert os.path.exists(f"./.model/{MODEL_FILENAME}"), "The reference weights does not exist. Please download it first (00-model-download.ipynb)"

# Loads the model weights
model_weight = torch.load(f"./.model/{MODEL_FILENAME}", map_location='cpu', weights_only=True, mmap=True)

# Model filename
print(f"### Model filename: {MODEL_FILENAME}")
print(model_weight.keys())

# Lets get the hidden_size, and setup the test module
head_size = 64
hidden_size = model_weight['model.embed_tokens.weight'].shape[1]
hidden_size_att = model_weight['model.layers.0.self_attn.v_proj.weight'].shape[0]
print(f"### Model hidden_size: {hidden_size}")

# List the model weights keys, and their shapes
print(f"### model weights keys:")
for key in model_weight:
    print(f"{key}: {model_weight[key].shape} - {model_weight[key].dtype}")

/home/harrison/.local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Model filename: rwkv-final.pth
odict_keys(['model.embed_tokens.weight', 'model.layers.0.input_layernorm.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.0.self_attn.time_maa_x', 'model.layers.0.self_attn.time_maa_r', 'model.layers.0.self_attn.time_maa_k', 'model.layers.0.self_attn.time_maa_v', 'model.layers.0.self_attn.time_maa_w', 'model.layers.0.self_attn.time_maa_g', 'model.layers.0.self_attn.time_maa_w2', 'model.layers.0.self_attn.time_maa_w1', 'model.layers.0.self_attn.time_decay', 'model.layers.0.self_attn.time_decay_w1', 'model.layers.0.self_attn.time_decay_w2', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.q_proj.bias', 'model.layers.0.self_attn.k_proj.weight', 'model.layers.0.self_attn.k_proj.bias', 'model.layers.0.self_attn.v_proj.weight', 'model.layers.0.self_attn.v_proj.bias', 'model.layers.0.self_attn.o_proj.weight', 'model.layers.0.self_attn.gate.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.w

In [2]:
# Initialize the channelmix state, and x state to test
#
# NOTE: The triton kernel minimum chunk size is 16, it fallsback to pytorch mode otherwise
# we intentionally DO not use a unit of 16, so the remainder pytorch code kicks in for triton
from rwkv_block.v6_qwerky.block.qwerky6_time_mix import Qwerky6TimeMix


IN_TOKENS_LEN=9000
x_state_0 = torch.ones(1, IN_TOKENS_LEN, hidden_size, device=RUN_DEVICE, dtype=RUN_DTYPE)
x_state_1 = torch.ones(1, IN_TOKENS_LEN, hidden_size, device=RUN_DEVICE, dtype=RUN_DTYPE)
x_state_2 = torch.ones(1, IN_TOKENS_LEN, hidden_size, device=RUN_DEVICE, dtype=RUN_DTYPE)
tmix_shift_0 = torch.ones(1, hidden_size, device=RUN_DEVICE, dtype=RUN_DTYPE)
tmix_shift_1 = torch.ones(1, hidden_size, device=RUN_DEVICE, dtype=RUN_DTYPE)
tmix_wkv_0 = torch.ones(1, hidden_size // head_size, head_size, head_size, device=RUN_DEVICE, dtype=torch.float)
tmix_wkv_1 = torch.ones(1, hidden_size // head_size, head_size, head_size, device=RUN_DEVICE, dtype=torch.float)

# Iteration to test
TEST_STEPS = 5
if RUN_DEVICE != "cpu":
    TEST_STEPS=50

# Build the cmix block
tmix = Qwerky6TimeMix({ 
    "num_hidden_layers":27,
    "head_size":head_size,
    "hidden_size":hidden_size, 
    "hidden_size_att":hidden_size_att, 
    "layer_id":0, 
    "device":RUN_DEVICE, "dtype":RUN_DTYPE, "tmix_backend":RUN_TMIX_BACKEND 
})
tmix.load_from_model_state_dict(model_weight, 0)

# Get the named parameters
tmix_params = tmix.named_parameters()
print(f"### tmix named parameters:")
for name, param in tmix_params:
    print(f"{name}: {param.shape} - {param.dtype} - {param.device.type}")

# Log each item shape
tmix_state = tmix.state_dict()
print(f"### tmix state keys:")
for key in tmix_state:
    print(f"tmix.{key}: {tmix_state[key].shape} - {tmix_state[key].dtype} - {param.device.type}")
print("----")

### tmix named parameters:
time_maa_r: torch.Size([1, 1, 3584]) - torch.bfloat16 - cpu
time_maa_w: torch.Size([1, 1, 3584]) - torch.bfloat16 - cpu
time_maa_k: torch.Size([1, 1, 3584]) - torch.bfloat16 - cpu
time_maa_v: torch.Size([1, 1, 3584]) - torch.bfloat16 - cpu
time_maa_a: torch.Size([1, 1, 3584]) - torch.bfloat16 - cpu
time_maa_g: torch.Size([1, 1, 3584]) - torch.bfloat16 - cpu
time_maa_x: torch.Size([1, 1, 3584]) - torch.bfloat16 - cpu
time_maa_w2: torch.Size([5, 96, 3584]) - torch.bfloat16 - cpu
time_maa_w1: torch.Size([3584, 480]) - torch.bfloat16 - cpu
time_decay: torch.Size([1, 1, 3584]) - torch.bfloat16 - cpu
time_decay_w1: torch.Size([3584, 64]) - torch.bfloat16 - cpu
time_decay_w2: torch.Size([64, 3584]) - torch.bfloat16 - cpu
q_proj.weight: torch.Size([3584, 3584]) - torch.bfloat16 - cpu
q_proj.bias: torch.Size([3584]) - torch.bfloat16 - cpu
k_proj.weight: torch.Size([512, 3584]) - torch.bfloat16 - cpu
k_proj.bias: torch.Size([512]) - torch.bfloat16 - cpu
v_proj.weight: 

In [3]:
### TMix
with torch.inference_mode():

    # This is a warmup
    t0 = time.time()
    out_x = x_state_0
    t_shift = tmix_shift_0
    t_wkv = tmix_wkv_0
    for i in range(TEST_STEPS):
        out_x, t_wkv, t_shift = tmix(x_state_1, tmix_wkv_1, tmix_shift_1)
    t2 = time.time()
    print(f'1 tmix forward passes (warmup): {(t2-t0)*1000/TEST_STEPS} ms ({RUN_DEVICE}, {RUN_DTYPE})')

    # The actual run
    t1 = time.time()
    out_x = x_state_0
    t_shift = tmix_shift_0
    t_wkv = tmix_wkv_0
    for i in range(TEST_STEPS):
        out_x, t_wkv, t_shift = tmix(x_state_1, tmix_wkv_1, tmix_shift_1)
    t2 = time.time()
    print(f'1 tmix forward passes (normal): {(t2-t1)*1000/TEST_STEPS} ms ({RUN_DEVICE}, {RUN_DTYPE})')


ValueError: Pointer argument (at 0) cannot be accessed from Triton (cpu tensor?)

In [4]:
### TMix
with torch.inference_mode():

    # This is a warmup
    t0 = time.time()
    out_x = x_state_0
    t_shift = tmix_shift_0
    t_wkv = tmix_wkv_0
    v_first = x_state_2
    for i in range(TEST_STEPS):
        out_x, t_wkv, v_first = tmix.forward_with_default_compile(x_state_1, tmix_wkv_1, v_first, out_x, t_wkv, v_first)
    t2 = time.time()
    print(f'1 tmix forward passes (warmup): {(t2-t0)*1000/TEST_STEPS} ms ({RUN_DEVICE}, {RUN_DTYPE})')

    # The actual run
    t1 = time.time()
    out_x = x_state_0
    t_shift = tmix_shift_0
    t_wkv = tmix_wkv_0
    v_first = x_state_2
    for i in range(TEST_STEPS):
        out_x, t_wkv, v_first = tmix.forward_with_default_compile(x_state_1, tmix_wkv_1, v_first, out_x, t_wkv, v_first)
    t2 = time.time()
    print(f'1 tmix forward passes (compiled): {(t2-t1)*1000/TEST_STEPS} ms ({RUN_DEVICE}, {RUN_DTYPE})')


1 tmix forward passes (warmup): 61.31931781768799 ms (cuda:0, torch.bfloat16)
1 tmix forward passes (compiled): 18.603901863098145 ms (cuda:0, torch.bfloat16)


In [5]:
### TMix
with torch.inference_mode():

    # This is a warmup
    t0 = time.time()
    out_x = x_state_0
    t_shift = tmix_shift_0
    t_wkv = tmix_wkv_0
    v_first = x_state_2
    for i in range(TEST_STEPS):
        out_x, t_wkv, v_first = tmix.forward_with_reduce_compile(x_state_1, tmix_wkv_1, v_first)
    t2 = time.time()
    print(f'1 tmix forward passes (warmup): {(t2-t0)*1000/TEST_STEPS} ms ({RUN_DEVICE}, {RUN_DTYPE})')

    # The actual run
    t1 = time.time()
    out_x = x_state_0
    t_shift = tmix_shift_0
    t_wkv = tmix_wkv_0
    v_first = x_state_2
    for i in range(TEST_STEPS):
        out_x, t_wkv, v_first = tmix.forward_with_reduce_compile(x_state_1, tmix_wkv_1, v_first)
    t2 = time.time()
    print(f'1 tmix forward passes (normal): {(t2-t1)*1000/TEST_STEPS} ms ({RUN_DEVICE}, {RUN_DTYPE})')


1 tmix forward passes (warmup): 52.87830352783203 ms (cuda:0, torch.bfloat16)
1 tmix forward passes (normal): 18.80061626434326 ms (cuda:0, torch.bfloat16)


In [6]:
# # Export tmix1 state dict
# tmix_state = tmix.state_dict()

# # Log each item shape
# print(f"### tmix state keys:")
# for key in tmix_state:
#     print(f"tmix.{key}: {tmix_state[key].shape} - {tmix_state[key].dtype}")
# print("----")

# # Build the tmix block
# tmix2 = Qwerky7TimeMix({ "num_hidden_layers":24, "hidden_size":hidden_size, "layer_id":1, "tmix_backend":"torch", "device":RUN_DEVICE, "dtype":RUN_DTYPE })

# # Load the state dict
# tmix2.load_from_model_state_dict(tmix_state, 1)

# # Log each item shape
# print(f"### tmix2 state keys:")
# for key in tmix_state:
#     print(f"tmix.{key}: {tmix_state[key].shape} - {tmix_state[key].dtype}")
# print("----")